# 06 — Live Jobs SQL ExportThis notebook picks up the output of `collect_live_jobs.py` (the standalone script thatcalls the JSearch API on demand) and loads it into a **separate** PostgreSQL table called`live_jobs`. This keeps the historical analysis table (`jobs`) untouched, while the**Live Job Market Feed** dashboard page always reflects whatever the script most recently collected.**How to refresh the live feed end-to-end:**1. Run `collect_live_jobs.py` from the project root — pulls fresh postings from the JSearch API2. Run this notebook — loads the new CSV into PostgreSQL3. Click **Refresh** in Power BI — the Live Job Market Feed page updates with the new data

In [1]:
import pandas as pdimport sqlalchemyimport osfrom pathlib import Pathfrom dotenv import load_dotenvBASE_DIR = Path.cwd()load_dotenv(BASE_DIR / ".env")live_df = pd.read_csv(BASE_DIR / "data" / "scraped" / "live_jobs.csv")print("Live jobs collected:", len(live_df))

Live jobs collected: 199

In [2]:
live_df["employment_type"] = live_df["employment_type"].astype(str).str.upper().fillna("UNKNOWN")live_df["is_remote_flag"]  = live_df["is_remote"].apply(lambda x: 1 if x == True else 0)live_df["location"]       = live_df["location"].astype(str).str.strip()print("Jobs marked remote:", live_df["is_remote_flag"].sum())print("Jobs marked non-remote:", (live_df["is_remote_flag"] == 0).sum())

Jobs marked remote: 8\nJobs marked non-remote: 191

In [3]:
live_df["location"].value_counts().head(10)

locationLondon               31United Kingdom       23Greater London       12Anywhere               8Manchester             8(blank)                6Glasgow                3Nottingham             2Barnet                 2Belfast                2Name: count, dtype: int64

In [4]:
DB_USER     = os.getenv("DB_USER", "postgres")DB_PASSWORD = os.getenv("DB_PASSWORD")DB_HOST     = os.getenv("DB_HOST", "localhost")DB_PORT     = os.getenv("DB_PORT", "5432")DB_NAME     = os.getenv("DB_NAME", "uk_jobs_db")engine = sqlalchemy.create_engine(    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")live_df.to_sql(    name="live_jobs",    con=engine,    if_exists="replace",    index=False)print("LIVE JOBS TABLE UPDATED")print("Rows now in live_jobs table:", len(live_df))

LIVE JOBS TABLE UPDATED\nRows now in live_jobs table: 199

In [5]:
pd.read_sql('''    SELECT role_category, COUNT(*) AS live_job_count    FROM live_jobs    GROUP BY role_category    ORDER BY live_job_count DESC;''', engine)

  role_category  live_job_count0   Ai Engineer              451   Data Analyst             402  Data Engineer              393  Data Scientist             384    Ml Engineer              37

**Output of this notebook:** A PostgreSQL table called `live_jobs`, refreshed each time `collect_live_jobs.py` is re-run and this notebook is re-executed. This is the data source for the **Live Job Market Feed** dashboard page in Power BI.This is currently a manual 3-step process. A natural next step would be scheduling steps 1–2 with a cron job / Task Scheduler and using Power BI's scheduled refresh for step 3.